# Two-stage LoRA + ATLAS and Early Layers

**Public leaderboard: 0.596**, up from **0.592** for [Nishant Dahal's two-stage LoRA version 2](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool?scriptVersionId=359887828). The gain is **+0.004**, or 0.4 percentage points. The scored run is [version 1 of this notebook](https://www.kaggle.com/code/josephadamski91/two-stage-lora-atlas-and-early-layers?scriptVersionId=360008790).

The original two LoRA classifiers, n-gram model, centered twins, ensemble weights and final balancing are unchanged. Two sets of probabilities are added before the original balancing:

```python
P = 0.80 * P_original + 0.10 * P_atlas + 0.10 * P_early
```

ATLAS columns are reordered to match the baseline's class order. The original 100 balancing iterations then run unchanged. The 0.596 result measures both additions together; it does not isolate their individual contributions.

The ATLAS adapter uses Qwen3-8B-Base and was trained for five epochs on the official 2,699 rows. It scores each passage using source-conditioned token likelihoods. These are the same adapter weights used in the earlier blends.

The early-layer classifier uses mean hidden states from layers 1, 2 and 3 of frozen Qwen3-4B-Base, with masked pooling and a 512-token limit. Four decoder layers are loaded so the selected states precede the final RMSNorm. A StandardScaler and logistic regression (C=0.0025) are fitted on the official 2,699 rows. Test features use that training scaler. No transformer weights are trained in this notebook.

Both additions use only the official training rows. The original two-stage ensemble also uses a pool of 34,940 public passages, so this notebook as a whole is not trained only on official data. The original author's training details and data credits are retained below.

Run with two T4 GPUs and internet disabled. Output: `submission.parquet`. Inputs include Qwen3-8B-Base, Qwen3-14B-Base and Qwen3-4B-Base version 1, the [two-stage checkpoints, version 1](https://www.kaggle.com/datasets/nishantdahal/ntl-two-stage-lora/versions/1), and the [ATLAS adapter, version 2](https://www.kaggle.com/datasets/josephadamski91/ntl-atlas-qwen3-8b-v1/versions/2).

Credit to Nishant Dahal for the base notebook and two-stage checkpoints, Ko et al. for the [ATLAS method](https://arxiv.org/abs/2610.19583), and Aleksey Osipenko for the [early-layer comparison](https://www.kaggle.com/competitions/name-that-llm/discussion/750462) that prompted the added classifier. This ATLAS adapter is an independent implementation. Upstream data credits and component licenses are retained below.


In [ ]:
import gc, glob, json, os, re, time
import numpy as np
import pandas as pd
import torch
from torch import nn

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).
W_SVC = 1.021
LORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for
    "lora8b_v1":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type="qwen3", hidden_size=4096, num_hidden_layers=36)),
    "lora14b_v1": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
}
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin

# Column order of the checkpoints' heads (checked against each checkpoint's meta.json below).
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))   # every member adds w * log p here
print("train", train.shape, "test", test.shape, flush=True)


## 1. Original n-gram model

The original character, word, shape and function-word features and class-balanced SVM are retained. It fits on the official rows and the attached public pool.


In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]


def shape_text(t):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def function_text(t):
    return " ".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))


pool = pd.read_parquet(find("pool_v1.parquet"))
fit_texts = train.text.astype(str).tolist() + pool.text.tolist()
fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]
A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else function_text if view == "function" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "function" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test.text]) * weight)
svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)
P_svc = softmax(svc.decision_function(hstack(B, format="csr")))
LOGIT += W_SVC * np.log(np.clip(P_svc, 1e-9, 1))
del A, B, svc, pool
gc.collect()
print(f"n-gram model done ({time.time() - T0:.0f}s)", flush=True)


## 2. Original LoRA classifiers

Load the original 8B and 14B two-stage checkpoints. Their layer selection, pooling, class order, learned heads and inference settings are unchanged.


In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """First `keep` decoder layers, split over both T4s when it does not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    hd = getattr(mc, "head_dim", mc.hidden_size // mc.num_attention_heads)
    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd
               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." in k:
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z (n, 8 * hidden) and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    ids = [x + [tok.eos_token_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together = less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()
    del model
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}


## 3. Original centered twins

The original topic clustering and centered-feature heads run unchanged.


In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
np.fill_diagonal(D, 0)
groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance") \
    if len(test) >= 2 else np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
del D
print(f"topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18", flush=True)

for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
    LOGIT += mc["w"] * np.log(np.clip(softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), 1e-9, 1))
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(groups[twin_rows]):
            Zc[groups == g] -= Zc[groups == g].mean(0)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        LOGIT[twin_rows] += mc["twin_w"] * np.log(np.clip(softmax(centred), 1e-9, 1))
    del Z
    gc.collect()
    print(f"{ckpt_name} done ({time.time() - T0:.0f}s)", flush=True)


## 4. Additional probabilities and final blend

Compute the added probabilities below, blend them with the complete original ensemble, then run the unchanged class balancing and write the submission.


In [ ]:
"""Frozen Qwen3-4B early features and the previously tested official-data classifier."""
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


@torch.no_grad()
def early_features(texts, model, tokenizer, batch_size=8):
    # hidden_states[1:4] precede the final RMSNorm. Keep four decoder layers,
    # so the third selected state is not the normalized final output.
    out = np.zeros((len(texts), 3 * model.config.hidden_size), np.float32)
    order = np.argsort([len(t) for t in texts])
    device = next(model.parameters()).device
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        batch = [(tokenizer.bos_token or '') + texts[int(i)] for i in idx]
        enc = tokenizer(batch, return_tensors='pt', padding=True, truncation=True,
                        max_length=512, add_special_tokens=False).to(device)
        states = model(**enc, output_hidden_states=True).hidden_states
        if not all(torch.isfinite(states[layer]).all() for layer in (1, 2, 3)):
            model.to(torch.bfloat16)
            states = model(**enc, output_hidden_states=True).hidden_states
        parts = []
        for layer in (1, 2, 3):
            mask = enc['attention_mask'].float().unsqueeze(-1).to(states[layer].device)
            pooled = (states[layer].float() * mask).sum(1) / mask.sum(1).clamp(min=1)
            assert torch.isfinite(pooled).all(), 'Nonfinite early-layer features'
            parts.append(pooled.cpu().numpy())
        out[idx] = np.concatenate(parts, axis=1)
        del states, enc, parts
        if start % 256 == 0:
            print(f'early features {min(start + batch_size, len(texts))}/{len(texts)}', flush=True)
    return out


def fit_early_probabilities(train_features, test_features, targets, class_count):
    assert np.isfinite(train_features).all() and np.isfinite(test_features).all()
    scaler = StandardScaler().fit(train_features)
    classifier = LogisticRegression(C=0.0025, solver='lbfgs', max_iter=1200, tol=1e-4)
    classifier.fit(scaler.transform(train_features), targets)
    assert np.array_equal(classifier.classes_, np.arange(class_count))
    probabilities = classifier.predict_proba(scaler.transform(test_features))
    assert probabilities.shape == (len(test_features), class_count)
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(1), 1, atol=1e-6)
    print('early classifier iterations:', classifier.n_iter_.tolist(), flush=True)
    return probabilities


def predict_early():
    base = find_base(dict(model_type='qwen3', hidden_size=2560))
    config = AutoConfig.from_pretrained(base)
    config.num_hidden_layers = 4
    if getattr(config, 'layer_types', None):
        config.layer_types = config.layer_types[:4]
    tokenizer = AutoTokenizer.from_pretrained(base)
    tokenizer.padding_side = 'right'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModel.from_pretrained(base, config=config, dtype=torch.float16,
                                      device_map='auto').eval()
    tr = early_features(train.text.astype(str).tolist(), model, tokenizer)
    te = early_features(test.text.astype(str).tolist(), model, tokenizer)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return fit_early_probabilities(tr, te, y, K)


EARLY_PROB = predict_early()


In [ ]:
# The ATLAS class order differs from the baseline's classifier heads.
import pathlib
import sys as _sys
import hashlib as _hashlib
_atlas_roots = list(pathlib.Path(INPUT_DIR).rglob('atlas_bundle.json'))
assert len(_atlas_roots) == 1, 'Attach exactly one ATLAS bundle'
_atlas_root = _atlas_roots[0].parent
_atlas_meta = json.loads(_atlas_roots[0].read_text())
assert _atlas_meta['files']['adapters.safetensors'] == '29e9b32e09679ae4e3b2cdf0729c7011ecee730fdb0f20982593af0b40806106'
for _name, _expected in _atlas_meta['files'].items():
    with (_atlas_root / _name).open('rb') as _handle:
        assert _hashlib.file_digest(_handle, 'sha256').hexdigest() == _expected, _name
_sys.path.insert(0, str(_atlas_root))
from ntl_atlas.inference import predict as _atlas_predict
from ntl_rlcd.data import LABELS as _atlas_labels
assert len(set(LABELS)) == len(set(_atlas_labels)) == K
assert set(LABELS) == set(_atlas_labels)
_ap = _atlas_predict(test.text.tolist(),
    find_base(dict(model_type='qwen3', hidden_size=4096, num_hidden_layers=36)),
    _atlas_root, batch_size=1)
ATLAS_PROB = _ap[:, [_atlas_labels.index(label) for label in LABELS]]
assert ATLAS_PROB.shape == (len(test), K)
assert np.isfinite(ATLAS_PROB).all() and (ATLAS_PROB >= 0).all()
assert np.allclose(ATLAS_PROB.sum(1), 1, atol=1e-6)


In [ ]:
P = softmax(LOGIT)
assert EARLY_PROB.shape == P.shape == ATLAS_PROB.shape
assert np.isfinite(EARLY_PROB).all() and (EARLY_PROB >= 0).all()
assert np.allclose(EARLY_PROB.sum(1), 1, atol=1e-6)
P = 0.80 * P + 0.10 * ATLAS_PROB + 0.10 * EARLY_PROB
P /= P.sum(1, keepdims=True)
if len(test) >= 50:   # skip on the 3-row public test.csv
    target = np.full(K, len(P) / K)
    for _ in range(100):
        P *= target / P.sum(0)
        P /= P.sum(1, keepdims=True)
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")


## How the checkpoints were trained

Both checkpoints use the same recipe. It is the public "LoRA24" recipe with one change: a first stage on public data.

- **Model:** Qwen3-8B-Base (first 24 layers) or Qwen3-14B-Base (first 27 layers), LoRA rank 16, alpha 32, dropout 0.05 on every linear layer, 8 pooled layers with a LayerNorm each, two linear heads (plain and centred).
- **Batches:** triplets of passages that answer the same prompt, 6 triplets per step. The loss is the cross-entropy of the plain head plus the cross-entropy of the centred head on triplet-centred features, which teaches the model to tell apart answers to *the same* question.
- **Stage 1:** 1 epoch over the 34,940 public passages, class-weighted cross-entropy (the pool is unbalanced), AdamW lr 1e-4 (LoRA) / 1e-3 (heads), 10% warm-up then cosine.
- **Stage 2:** 2 epochs on the host's 2,699 passages, lr 5e-5 / 5e-4.
- Trained in bf16 on one RTX 5090: about 30 min (8B) and 90 min (14B) per checkpoint.

**The pool** merges 14 public datasets, keeping single-paragraph passages of 30-300 words with no markdown. Exact duplicates and anything identical to a training passage are dropped. One more filter: if a model *never* shows a mark in the training data (an em dash for Gemini, curly quotes for Claude), its pool passages containing that mark are dropped. That leaves 34,940 passages for all 18 models (1,100-2,700 per model).

## Credits

Public passages (pool), all CC BY 4.0 except where noted:
[seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages),
[willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest),
[lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1),
[prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset),
[prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1) (CC BY-SA 4.0),
[alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations),
[haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages),
[kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages),
[lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations),
[abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations),
[dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations),
[ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations),
[keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1),
[michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1).

Ideas and recipes:
the LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7);
the style-aware n-grams from [yangkuangou](https://www.kaggle.com/yangkuangou)'s notebook
adaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering).
Models: Qwen3-8B-Base and Qwen3-14B-Base (Qwen team, Apache 2.0).
